In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

TOKEN = ""
print("Imports successful ✓")

Imports successful ✓


In [5]:
def api(endpoint):
    r = requests.get(
        f"https://api.spotify.com/{endpoint}",
        headers={"Authorization": f"Bearer {TOKEN}"}
    )
    r.raise_for_status()
    return r.json()

# Quick check
me = api("v1/me")
print(f"Logged in as: {me['display_name']}")

Logged in as: Adron


In [10]:
TIME_RANGE = "long_term"   
POOL_SIZE  = 50
N_QUERIES  = 5
TOP_N      = 10

In [11]:
def fetch_top_tracks(time_range, limit):
    return api(f"v1/me/top/tracks?time_range={time_range}&limit={limit}").get("items", [])

def fetch_recommendations(seed_id, limit=50):
    return api(f"v1/recommendations?seed_tracks={seed_id}&limit=50").get("tracks", [])

top_tracks = fetch_top_tracks(TIME_RANGE, POOL_SIZE)
print(f"Loaded {len(top_tracks)} top tracks")
for i, t in enumerate(top_tracks[:5]):
    print(f"  {i+1}. {t['name']} — {t['artists'][0]['name']}")

Loaded 50 top tracks
  1. MUSTARD (WITH 6LACK) — Jordan Ward
  2. So Far Gone / Fast Life Bluez — Brent Faiyaz
  3. PIMPIN AINT EASY (feat. Lil Yachty) — Concrete Boys
  4. Over My Dead Body — Drake
  5. Fell In Love — Marshmello


In [12]:
# Build Track Pool — no recommendations endpoint, use medium + short term instead
pool = list(top_tracks)
seen = {t["id"] for t in pool}

# Widen pool with medium and short term top tracks
for time_range in ["medium_term", "short_term"]:
    extra = api(f"v1/me/top/tracks?time_range={time_range}&limit=50").get("items", [])
    for t in extra:
        if t["id"] not in seen:
            pool.append(t)
            seen.add(t["id"])

print(f"Pool size: {len(pool)} unique tracks")
print("Sample:", [t["name"] for t in pool[:5]])

Pool size: 50 unique tracks
Sample: ['MUSTARD (WITH 6LACK)', 'So Far Gone / Fast Life Bluez', 'PIMPIN AINT EASY (feat. Lil Yachty)', 'Over My Dead Body', 'Fell In Love']


In [13]:
# Audio features blocked — use track metadata instead
rows = []

for i in range(0, len(pool), 50):
    batch = pool[i:i+50]
    ids = ",".join(t["id"] for t in batch)
    data = api(f"v1/tracks?ids={ids}")
    
    for track in data["tracks"]:
        if not track:
            continue
        rows.append({
            "id":              track["id"],
            "track":           track["name"],
            "artist":          track["artists"][0]["name"],
            "popularity":      track["popularity"],              # 0-100
            "duration_ms":     track["duration_ms"],            # milliseconds
            "explicit":        int(track["explicit"]),          # 0 or 1
            "n_artists":       len(track["artists"]),           # collaboration proxy
            "release_year":    int(track["album"]["release_date"][:4]),
            "is_single":       int(track["album"]["album_type"] == "single"),
        })

df = pd.DataFrame(rows).drop_duplicates("track").reset_index(drop=True)
print(f"Tracks loaded: {len(df)}")
print(df[["track", "artist", "popularity", "duration_ms", "release_year"]].head(10))

Tracks loaded: 50
                                             track         artist  popularity  \
0                             MUSTARD (WITH 6LACK)    Jordan Ward          35   
1                    So Far Gone / Fast Life Bluez   Brent Faiyaz          59   
2              PIMPIN AINT EASY (feat. Lil Yachty)  Concrete Boys           0   
3                                Over My Dead Body          Drake          79   
4                                     Fell In Love     Marshmello          58   
5                                   Tried Our Best          Drake          73   
6  Trillionaire (feat. Youngboy Never Broke Again)         Future          60   
7                                        Come Thru          Drake          68   
8                                       U With Me?          Drake          67   
9           Law Of Attraction (feat. Snoh Aalegra)           Dave          35   

   duration_ms  release_year  
0       201464          2023  
1       331018          2017

In [14]:
FEATURE_KEYS = [
    "popularity", "duration_ms", "explicit",
    "n_artists", "release_year", "is_single"
]

scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURE_KEYS])
sim_matrix = cosine_similarity(X)

print(f"Matrix shape: {sim_matrix.shape}")
print(f"Diagonal check (should be 1.0): {sim_matrix.diagonal()[:3].round(4)}")

Matrix shape: (50, 50)
Diagonal check (should be 1.0): [1. 1. 1.]


In [15]:
# Standardize Features 
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURE_KEYS])
sim_matrix = cosine_similarity(X)

print(f"Matrix shape: {sim_matrix.shape}")
print(f"Diagonal check (should be 1.0): {sim_matrix.diagonal()[:3].round(4)}")

Matrix shape: (50, 50)
Diagonal check (should be 1.0): [1. 1. 1.]


In [16]:
# Similarity matrix

def find_similar(query_idx, top_n=10):
    scores = list(enumerate(sim_matrix[query_idx]))
    scores.sort(key=lambda x: x[1], reverse=True)
    results = [(df.iloc[i]["track"], df.iloc[i]["artist"], round(s, 4))
               for i, s in scores if i != query_idx][:top_n]
    return results

query_results = {}
for qi in range(min(N_QUERIES, len(df))):
    query_results[qi] = find_similar(qi, TOP_N)

print(f"Done — results computed for {len(query_results)} queries.")

Done — results computed for 5 queries.


In [17]:
# Display results 
for qi, results in query_results.items():
    qtrack = df.iloc[qi]
    print(f"\n{'─'*55}")
    print(f"Query: {qtrack['track']} — {qtrack['artist']}")
    print(f"{'─'*55}")
    print(f"{'#':<4} {'Track':<35} {'Artist':<20} {'Sim':>6}")
    print(f"{'─'*55}")
    for rank, (track, artist, score) in enumerate(results, 1):
        print(f"{rank:<4} {track[:34]:<35} {artist[:19]:<20} {score:>6.3f}")


───────────────────────────────────────────────────────
Query: MUSTARD (WITH 6LACK) — Jordan Ward
───────────────────────────────────────────────────────
#    Track                               Artist                  Sim
───────────────────────────────────────────────────────
1    act i: stickerz "99"                4batz                 0.755
2    Dear April (Side A - Acoustic)      Frank Ocean           0.684
3    Don't Look At Numbers - Remix       Tony Shhnow           0.627
4    Our 25th Birthday                   Dave                  0.556
5    HVN ON EARTH (with Kodak Black)     Lil Tecca             0.520
6    One Night Only                      Sonder                0.490
7    Her                                 Majid Jordan          0.475
8    Outta Time (feat. Drake)            Bryson Tiller         0.473
9    Fell In Love                        Marshmello            0.470
10   My Affection (with PARTYNEXTDOOR)   Summer Walker         0.468

─────────────────────────────